# 01 - Scraping Data Inflasi Indonesia
Notebook ini berisi proses pengambilan data dari berbagai sumber (IMF, FRED, TradingEconomics) secara otomatis.

In [9]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv

# Add src to path
sys.path.append('../')

load_dotenv('../.env')
os.makedirs('../data/raw', exist_ok=True)

## 1. IMF Scraping (Inflasi & Reserves)
Menggunakan REST API publik IMF.

In [10]:
from src.scraper.imf_scraper import scrape_imf

print("Scraping IMF data...")
imf_inf = scrape_imf("PCPIPCH")
if not imf_inf.empty:
    imf_inf.to_csv('../data/raw/imf_inflation.csv', index=False)
    display(imf_inf.head())

Scraping IMF data...
✅ IMF PCPIPCH (IDN): 613 rows


,date,pcpipch
0,1980-01-01,18.0
1,1980-02-01,18.0
2,1980-03-01,18.0
3,1980-04-01,18.0
4,1980-05-01,18.0


## 2. FRED Scraping (Macro Indicators)
Membutuhkan API Key di file `.env`.

In [11]:
from src.scraper.fred_scraper import scrape_fred, FRED_SERIES

all_fred = []
for series_id, col_name in FRED_SERIES.items():
    print(f"Scraping FRED: {series_id} -> {col_name}")
    df = scrape_fred(series_id, start_date="2010-01-01")
    if not df.empty:
        df.rename(columns={series_id.lower(): col_name}, inplace=True)
        all_fred.append(df)

if all_fred:
    from functools import reduce
    fred_combined = reduce(lambda l, r: pd.merge(l, r, on='date', how='outer'), all_fred)
    fred_combined.to_csv('../data/raw/fred_data.csv', index=False)
    display(fred_combined.head())
else:
    print("No FRED data collected. Check your API Key in .env")

Scraping FRED: DEXINUS -> usd_idr
⚠️ FRED_API_KEY not set. Skipping DEXINUS
Scraping FRED: FEDFUNDS -> fed_fund_rate
⚠️ FRED_API_KEY not set. Skipping FEDFUNDS
Scraping FRED: DCOILBRENTEU -> oil_price_brent
⚠️ FRED_API_KEY not set. Skipping DCOILBRENTEU
Scraping FRED: IDNCPIALLMINMEI -> inflation_yoy
⚠️ FRED_API_KEY not set. Skipping IDNCPIALLMINMEI
No FRED data collected. Check your API Key in .env


## 3. TradingEconomics Scraping

In [12]:
from src.scraper.te_scraper import scrape_te_inflation

te_df = scrape_te_inflation()
if not te_df.empty:
    te_df.to_csv('../data/raw/te_inflation.csv', index=False)
    display(te_df.head())
else:
    print("Scraping TE failed or returned empty.")

❌ TE Scraping failed: [Errno 2] No such file or directory: 
<!doctype html>
<html >
<head id="ctl00_Head1"><meta charset="utf-8" /><title>
	Indonesia Inflation Rate
</title><meta id="metaDesc" name="description" content="Inflation Rate in Indonesia decreased to 2.42 percent in April from 3.48 percent in March of 2026. This page provides - Indonesia Inflation Rate - actual values, historical data, forecast, chart, statistics, economic calendar and news." /><meta id="metaKeyword" name="keywords" content="Indonesia,Inflation Rate,data,chart,actual,historical,values,calendar,forecast,graph,2025,2026,2027" /><meta name="viewport" content="width=device-width,minimum-scale=1,initial-scale=1,maximum-scale=1" /><meta name="theme-color" content="#333333" /><link rel="preconnect" href="https://d3e5kp2e91w0k0.cloudfront.net" crossorigin="anonymous" /><link rel="preconnect" href="https://d2n2sd20z6hutz.cloudfront.net" crossorigin="anonymous" /><link rel="preconnect" href="https://cdnjs.cloudflare.c